In [1]:
%pip install torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split

LABELS = [
    "Hate",
    "Constructive_criticism",
    "Appreciation",
    "Question",
    "Sarcasm",
    "Random_unrelated"
]

print("PyTorch version:", torch.__version__)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")

PyTorch version: 2.15.0+cpu
Device: CPU


In [3]:
df = pd.read_csv("dataset/youtube_comments_1000_final_labeled.csv")

modelDf = df[["CommentText"] + LABELS].copy()

print(modelDf.shape)
modelDf.head()

(1000, 7)


,CommentText,Hate,Constructive_criticism,Appreciation,Question,Sarcasm,Random_unrelated
0,"1:30 actually, ""kek"" goes all the way back *tw...",0,0,0,0,1,0
1,right away i get the feeling that you lay thin...,0,0,1,0,0,0
2,Didi please technical questions pe videos banao,0,0,0,1,0,0
3,Sold high ticket stuff for 40+/yrs...learned e...,0,0,0,0,0,0
4,I doubt even a brand new bridge could withstan...,0,1,0,0,0,0


In [4]:
X = modelDf["CommentText"]
y = modelDf[LABELS]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1000,)
y shape: (1000, 6)


In [5]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# First split: 70% training, 30% temporary
split1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

trainIdx, tempIdx = next(split1.split(X, y))

XTrain = X.iloc[trainIdx]
yTrain = y.iloc[trainIdx]

XTemp = X.iloc[tempIdx]
yTemp = y.iloc[tempIdx]


# Second split: 15% validation, 15% test
split2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

valIdx, testIdx = next(split2.split(XTemp, yTemp))

valIdx, testIdx = next(split2.split(XTemp, yTemp))

XVal = XTemp.iloc[valIdx]
yVal = yTemp.iloc[valIdx]

XTest = XTemp.iloc[testIdx]
yTest = yTemp.iloc[testIdx]


In [6]:
print("Training positives:")
print(yTrain.sum())

print("\nValidation positives:")
print(yVal.sum())

print("\nTest positives:")
print(yTest.sum())

print("Training percentages:")
print(yTrain.mean() * 100)

Training positives:
Hate                       37
Constructive_criticism    226
Appreciation              211
Question                   88
Sarcasm                   111
Random_unrelated           61
dtype: int64

Validation positives:
Hate                       8
Constructive_criticism    48
Appreciation              45
Question                  19
Sarcasm                   23
Random_unrelated          13
dtype: int64

Test positives:
Hate                       8
Constructive_criticism    49
Appreciation              46
Question                  19
Sarcasm                   24
Random_unrelated          13
dtype: int64
Training percentages:
Hate                       5.285714
Constructive_criticism    32.285714
Appreciation              30.142857
Question                  12.571429
Sarcasm                   15.857143
Random_unrelated           8.714286
dtype: float64


In [7]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

trainEncodings = tokenizer(
    XTrain.tolist(),
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)

# print(trainEncodings.keys())

# print("Input IDs:", trainEncodings["input_ids"].shape)
# print("Attention Mask:", trainEncodings["attention_mask"].shape)

# print(XTrain.iloc[0])
# print(tokenizer.convert_ids_to_tokens(
#     trainEncodings["input_ids"][0]
# ))

valEncodings = tokenizer(
    XVal.tolist(),
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)

testEncodings = tokenizer(
    XTest.tolist(),
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"
)

# print("Train:", trainEncodings["input_ids"].shape)
# print("Validation:", valEncodings["input_ids"].shape)
# print("Test:", testEncodings["input_ids"].shape)

c:\Users\Asus\Desktop\NLP With LLM\EndSemProject_CmntSentiAnalyser\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import torch
from torch.utils.data import Dataset

class CommentDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(
            labels.values,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):

        item = {
            key: value[index]
            for key, value in self.encodings.items()
            if key in ["input_ids", "attention_mask"]
        }

        item["labels"] = self.labels[index]

        return item

In [9]:
trainDataset = CommentDataset(trainEncodings, yTrain)
valDataset = CommentDataset(valEncodings, yVal)
testDataset = CommentDataset(testEncodings, yTest)

sample = trainDataset[0]

for key, value in sample.items():
    print(key, value.shape, value.dtype)
    

input_ids torch.Size([128]) torch.int64
attention_mask torch.Size([128]) torch.int64
labels torch.Size([6]) torch.float32


In [10]:
from torch.utils.data import DataLoader

batchSize = 8

trainLoader = DataLoader(
    trainDataset,
    batch_size=batchSize,
    shuffle=True
)

valLoader = DataLoader(
    valDataset,
    batch_size=batchSize,
    shuffle=False
)

testLoader = DataLoader(
    testDataset,
    batch_size=batchSize,
    shuffle=False
)

In [11]:
batch = next(iter(trainLoader))

for key, value in batch.items():
    print(key, value.shape, value.dtype)

input_ids torch.Size([8, 128]) torch.int64
attention_mask torch.Size([8, 128]) torch.int64
labels torch.Size([8, 6]) torch.float32


In [12]:
processedData = {
    "xTrain": XTrain,
    "yTrain": yTrain,
    "xVal": XVal,
    "yVal": yVal,
    "xTest": XTest,
    "yTest": yTest,
    "labels": LABELS
}

pd.to_pickle(processedData, "dataset/processedSplits.pkl")